# MIDI comparison for game, classical, and pop music

This notebook compares broad musical features across three corpora:
- `game_midi`
- `classical_midi`
- `pop_midi`

The goal is to test whether game music tends to be sparser, more repetitive in a softer way, and harmonically less goal-directed than classical or pop music.

In [1]:
from pathlib import Path
from collections import Counter
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pretty_midi

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

BASE_DIR = Path.cwd()
DATASETS = {
    'game': BASE_DIR / 'game_midi',
    'classical': BASE_DIR / 'classical_midi',
    'pop': BASE_DIR / 'pop_midi',
}

for label, folder in DATASETS.items():
    print(label, folder.exists(), folder)

game True /Users/margo/Desktop/game_music/game_midi
classical True /Users/margo/Desktop/game_music/classical_midi
pop True /Users/margo/Desktop/game_music/pop_midi


In [2]:
def collect_midi_files(folder):
    return sorted([p for p in folder.rglob('*') if p.suffix.lower() in {'.mid', '.midi'}])

rows = []
for genre, folder in DATASETS.items():
    for path in collect_midi_files(folder):
        rows.append({
            'genre': genre,
            'path': path,
            'filename': path.name,
        })

files_df = pd.DataFrame(rows)
files_df.groupby('genre').size().rename('file_count').to_frame()

,file_count
genre,
classical,295
game,4054
pop,50


## Loop Feature Helpers

These functions describe how game music is built to repeat: they look for repeated blocks, measure the space before a return, and check whether the ending harmony feels ready to cycle back into the opening.

In [3]:
PITCH_CLASS_NAMES = np.array(['C', 'C#', 'D', 'Eb', 'E', 'F', 'F#', 'G', 'Ab', 'A', 'Bb', 'B'])

def cosine_similarity(left, right):
    left_norm = float(np.linalg.norm(left))
    right_norm = float(np.linalg.norm(right))
    if left_norm == 0.0 or right_norm == 0.0:
        return 0.0
    return float(np.dot(left, right) / (left_norm * right_norm))

def collect_melodic_notes(pm):
    notes = []
    for inst in pm.instruments:
        if inst.is_drum:
            continue
        notes.extend(inst.notes)
    return notes

def get_beat_grid(pm):
    end_time = max(pm.get_end_time(), 1e-6)
    beats = np.asarray(pm.get_beats(), dtype=float)

    if beats.size < 2:
        tempi, _ = pm.get_tempo_changes()
        tempo = float(np.median(tempi)) if len(tempi) else 120.0
        beat_duration = 60.0 / max(tempo, 1e-6)
        beats = np.arange(0.0, end_time + beat_duration, beat_duration)

    beats = np.unique(np.clip(beats, 0.0, end_time))
    if beats.size == 0 or beats[0] > 0.0:
        beats = np.insert(beats, 0, 0.0)
    if beats[-1] < end_time:
        beats = np.append(beats, end_time)
    return beats

def get_window_edges(pm, beats_per_window=4):
    beats = get_beat_grid(pm)
    end_time = max(pm.get_end_time(), 1e-6)

    if beats.size <= beats_per_window:
        return np.array([0.0, end_time], dtype=float)

    edges = beats[::beats_per_window]
    if edges[0] != 0.0:
        edges = np.insert(edges, 0, 0.0)
    if edges[-1] < end_time:
        edges = np.append(edges, end_time)
    return np.unique(edges)

def window_overlap(note, start, end):
    return max(0.0, min(note.end, end) - max(note.start, start))

def window_signature(notes, start, end):
    chroma = np.zeros(12, dtype=float)
    onset_chroma = np.zeros(12, dtype=float)
    note_count = 0.0
    onset_count = 0.0
    pitch_sum = 0.0

    for note in notes:
        overlap = window_overlap(note, start, end)
        if overlap <= 0.0:
            continue
        pc = note.pitch % 12
        chroma[pc] += overlap
        note_count += 1.0
        pitch_sum += note.pitch
        if start <= note.start < end:
            onset_chroma[pc] += 1.0
            onset_count += 1.0

    window_duration = max(end - start, 1e-6)
    chroma = chroma / max(chroma.sum(), 1e-6)
    onset_chroma = onset_chroma / max(onset_chroma.sum(), 1e-6)
    pitch_center = pitch_sum / max(note_count, 1.0)
    extra = np.array([
        note_count / window_duration,
        onset_count / window_duration,
        pitch_center / 127.0,
    ], dtype=float)
    return np.concatenate([chroma, onset_chroma, extra]), chroma

def match_template(chroma):
    if chroma.sum() <= 0.0:
        return None, None, 0.0

    templates = []
    for root in range(12):
        major = np.zeros(12, dtype=float)
        major[[root, (root + 4) % 12, (root + 7) % 12]] = 1.0
        minor = np.zeros(12, dtype=float)
        minor[[root, (root + 3) % 12, (root + 7) % 12]] = 1.0
        templates.append((root, 'maj', major))
        templates.append((root, 'min', minor))

    best_root = None
    best_quality = None
    best_score = -1.0
    for root, quality, template in templates:
        score = cosine_similarity(chroma, template)
        if score > best_score:
            best_root = root
            best_quality = quality
            best_score = score

    if best_score < 0.45:
        return None, None, float(best_score)
    return best_root, best_quality, float(best_score)

def analyze_repeated_sections(signatures, edges, total_duration, min_windows=2, similarity_threshold=0.92):
    n_windows = len(signatures)
    if n_windows < min_windows * 2:
        return {
            'repeated_section_count': 0,
            'repeated_window_ratio': 0.0,
            'longest_repeated_span_sec': 0.0,
            'longest_repeated_span_ratio': 0.0,
            'repeat_gap_sec': float('nan'),
            'repeat_gap_ratio': float('nan'),
            'mean_repeat_similarity': float('nan'),
        }

    matches = []
    max_windows = n_windows // 2
    for block_len in range(min_windows, max_windows + 1):
        for left in range(0, n_windows - (2 * block_len) + 1):
            for right in range(left + block_len, n_windows - block_len + 1):
                sims = [
                    cosine_similarity(signatures[left + offset], signatures[right + offset])
                    for offset in range(block_len)
                ]
                block_similarity = float(np.mean(sims))
                if block_similarity < similarity_threshold:
                    continue
                left_start = float(edges[left])
                left_end = float(edges[left + block_len])
                right_start = float(edges[right])
                matches.append({
                    'similarity': block_similarity,
                    'length_sec': left_end - left_start,
                    'gap_sec': right_start - left_end,
                    'covered': set(range(left, left + block_len)) | set(range(right, right + block_len)),
                })

    if not matches:
        return {
            'repeated_section_count': 0,
            'repeated_window_ratio': 0.0,
            'longest_repeated_span_sec': 0.0,
            'longest_repeated_span_ratio': 0.0,
            'repeat_gap_sec': float('nan'),
            'repeat_gap_ratio': float('nan'),
            'mean_repeat_similarity': float('nan'),
        }

    best = max(matches, key=lambda item: (item['length_sec'], item['similarity']))
    covered_windows = set()
    for match in matches:
        covered_windows.update(match['covered'])

    return {
        'repeated_section_count': int(len(matches)),
        'repeated_window_ratio': float(len(covered_windows) / n_windows),
        'longest_repeated_span_sec': float(best['length_sec']),
        'longest_repeated_span_ratio': float(best['length_sec'] / total_duration),
        'repeat_gap_sec': float(best['gap_sec']),
        'repeat_gap_ratio': float(best['gap_sec'] / total_duration),
        'mean_repeat_similarity': float(np.mean([match['similarity'] for match in matches])),
    }

def extract_loop_features(path_or_pm, beats_per_window=4):
    pm = path_or_pm if isinstance(path_or_pm, pretty_midi.PrettyMIDI) else pretty_midi.PrettyMIDI(str(path_or_pm))
    notes = collect_melodic_notes(pm)
    total_duration = max(pm.get_end_time(), 1e-6)

    if not notes:
        return {
            'loop_window_count': 0,
            'repeated_section_count': 0,
            'repeated_window_ratio': 0.0,
            'longest_repeated_span_sec': 0.0,
            'longest_repeated_span_ratio': 0.0,
            'repeat_gap_sec': float('nan'),
            'repeat_gap_ratio': float('nan'),
            'mean_repeat_similarity': float('nan'),
            'chord_loop_closure_similarity': float('nan'),
            'two_step_loop_similarity': float('nan'),
            'closing_matches_opening_chord': 0.0,
            'closing_contains_opening_root': 0.0,
            'penultimate_to_opening_fifth_resolution': 0.0,
            'opening_chord': 'NA',
            'closing_chord': 'NA',
            'window_chord_repetition_ratio': float('nan'),
        }

    edges = get_window_edges(pm, beats_per_window=beats_per_window)
    signatures = []
    chroma_windows = []
    for start, end in zip(edges[:-1], edges[1:]):
        signature, chroma = window_signature(notes, float(start), float(end))
        signatures.append(signature)
        chroma_windows.append(chroma)

    repeat_features = analyze_repeated_sections(signatures, edges, total_duration)
    first_chroma = chroma_windows[0]
    last_chroma = chroma_windows[-1]
    penultimate_chroma = chroma_windows[-2] if len(chroma_windows) >= 2 else np.zeros(12, dtype=float)
    chord_matches = [match_template(chroma) for chroma in chroma_windows]

    opening_root, opening_quality, _ = match_template(first_chroma)
    closing_root, closing_quality, _ = match_template(last_chroma)
    penultimate_root, _, _ = match_template(penultimate_chroma)

    opening_label = f'{PITCH_CLASS_NAMES[opening_root]}:{opening_quality}' if opening_root is not None and opening_quality else 'unknown'
    closing_label = f'{PITCH_CLASS_NAMES[closing_root]}:{closing_quality}' if closing_root is not None and closing_quality else 'unknown'

    closing_contains_opening_root = 0.0
    if opening_root is not None and last_chroma.sum() > 0.0:
        closing_contains_opening_root = float(
            last_chroma[opening_root] >= np.percentile(last_chroma[last_chroma > 0], 50)
        ) if np.any(last_chroma > 0) else 0.0

    penultimate_to_opening = 0.0
    if opening_root is not None and penultimate_root is not None:
        penultimate_to_opening = float((penultimate_root - opening_root) % 12 == 7)

    two_step_similarity = float('nan')
    if len(chroma_windows) >= 2:
        two_step_similarity = float(np.mean([
            cosine_similarity(penultimate_chroma, first_chroma),
            cosine_similarity(last_chroma, first_chroma),
        ]))

    return {
        'loop_window_count': int(len(signatures)),
        **repeat_features,
        'chord_loop_closure_similarity': float(cosine_similarity(first_chroma, last_chroma)),
        'two_step_loop_similarity': float(two_step_similarity),
        'closing_matches_opening_chord': float(
            opening_root is not None and closing_root is not None and opening_root == closing_root and opening_quality == closing_quality
        ),
        'closing_contains_opening_root': closing_contains_opening_root,
        'penultimate_to_opening_fifth_resolution': penultimate_to_opening,
        'opening_chord': opening_label,
        'closing_chord': closing_label,
        'window_chord_repetition_ratio': float(
            repetition_ratio(tuple((root, quality) for root, quality, _ in chord_matches if root is not None))
        ),
    }


In [4]:
def pitch_class_entropy(pitches):
    if len(pitches) == 0:
        return np.nan
    counts = np.bincount(np.array(pitches) % 12, minlength=12).astype(float)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())

def rhythmic_entropy(onsets, decimals=2):
    if len(onsets) < 3:
        return np.nan
    ioi = np.diff(np.sort(onsets))
    ioi = ioi[ioi > 0]
    if len(ioi) == 0:
        return np.nan
    rounded = np.round(ioi, decimals)
    values, counts = np.unique(rounded, return_counts=True)
    probs = counts / counts.sum()
    return float(-(probs * np.log2(probs)).sum())

def repetition_ratio(items):
    if len(items) == 0:
        return np.nan
    counts = Counter(items)
    repeated = sum(count for count in counts.values() if count > 1)
    return repeated / len(items)

def estimate_chords(pm, window=1.0):
    end_time = pm.get_end_time()
    if end_time <= 0:
        return []

    chords = []
    starts = np.arange(0, end_time, window)
    for start in starts:
        end = start + window
        active = []
        for inst in pm.instruments:
            if inst.is_drum:
                continue
            for note in inst.notes:
                if note.start < end and note.end > start:
                    active.append(note.pitch % 12)
        chord = tuple(sorted(set(active)))
        if len(chord) >= 2:
            chords.append(chord)
    return chords

def extract_features(path):
    try:
        pm = pretty_midi.PrettyMIDI(str(path))
    except Exception:
        return None

    melodic_notes = []
    drum_notes = []
    all_onsets = []
    durations = []
    velocities = []
    pitch_classes = []
    instrument_programs = set()

    for inst in pm.instruments:
        if not inst.is_drum:
            instrument_programs.add(inst.program)
        for note in inst.notes:
            if inst.is_drum:
                drum_notes.append(note)
            else:
                melodic_notes.append(note)
                pitch_classes.append(note.pitch % 12)
            all_onsets.append(note.start)
            durations.append(note.end - note.start)
            velocities.append(note.velocity)

    if len(melodic_notes) == 0:
        return None

    end_time = max(pm.get_end_time(), 1e-6)
    note_starts = np.array(sorted([n.start for n in melodic_notes]))
    pitches = np.array([n.pitch for n in melodic_notes])
    durations = np.array(durations)
    velocities = np.array(velocities)

    ioi = np.diff(note_starts)
    positive_ioi = ioi[ioi > 0]
    chords = estimate_chords(pm, window=1.0)
    chord_changes = 0
    if len(chords) > 1:
        chord_changes = sum(chords[i] != chords[i - 1] for i in range(1, len(chords)))

    tempo_estimates, beat_times = pm.get_tempo_changes()
    estimated_tempo = float(np.median(tempo_estimates)) if len(tempo_estimates) else np.nan
    loop_features = extract_loop_features(pm)

    return {
        'duration_sec': end_time,
        'n_melodic_notes': len(melodic_notes),
        'n_drum_notes': len(drum_notes),
        'note_density': len(melodic_notes) / end_time,
        'drum_density': len(drum_notes) / end_time,
        'silence_proxy': 1 - min(len(np.unique(np.round(note_starts, 2))) / max(end_time, 1), 1),
        'avg_note_duration': float(np.mean([n.end - n.start for n in melodic_notes])),
        'duration_std': float(np.std([n.end - n.start for n in melodic_notes])),
        'avg_velocity': float(np.mean(velocities)) if len(velocities) else np.nan,
        'velocity_std': float(np.std(velocities)) if len(velocities) else np.nan,
        'pitch_range': int(np.max(pitches) - np.min(pitches)),
        'pitch_class_entropy': pitch_class_entropy(pitch_classes),
        'avg_ioi': float(np.mean(positive_ioi)) if len(positive_ioi) else np.nan,
        'ioi_std': float(np.std(positive_ioi)) if len(positive_ioi) else np.nan,
        'rhythmic_entropy': rhythmic_entropy(note_starts),
        'estimated_tempo': estimated_tempo,
        'instrument_variety': len(instrument_programs),
        'unique_pitch_classes': len(set(pitch_classes)),
        'chord_vocab_size': len(set(chords)),
        'chord_change_rate': chord_changes / end_time,
        'chord_repetition_ratio': repetition_ratio(chords),
        'pitch_repetition_ratio': repetition_ratio(list(pitches)),
        **loop_features,
    }


In [5]:
# If the full corpus is too slow, lower this number.
MAX_FILES_PER_GENRE = 300

sampled = (
    files_df.groupby('genre', group_keys=False)
    .head(MAX_FILES_PER_GENRE)
    .reset_index(drop=True)
)

print(sampled.groupby('genre').size())

feature_rows = []
for row in sampled.itertuples(index=False):
    features = extract_features(row.path)
    if features is None:
        continue
    features['genre'] = row.genre
    features['filename'] = row.filename
    features['path'] = str(row.path)
    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)
features_df.head()

genre
classical    295
game         300
pop           50
dtype: int64


KeyboardInterrupt: 

In [ ]:
numeric_features = features_df.select_dtypes(include=[np.number]).columns
summary = features_df.groupby('genre')[numeric_features].agg([
    'mean', 'median', 'std'
]).round(3)
summary

TypeError: dtype 'str' does not support operation 'mean'

## Loop-specific follow-up

These views focus on loop construction: how long repeated spans are, how much space sits between the original and its return, and whether the ending harmony feels ready to cycle back into the opening.

In [ ]:
loop_columns = [
    'longest_repeated_span_ratio',
    'repeat_gap_ratio',
    'repeated_window_ratio',
    'mean_repeat_similarity',
    'chord_loop_closure_similarity',
    'closing_matches_opening_chord',
    'closing_contains_opening_root',
    'penultimate_to_opening_fifth_resolution',
]

features_df.groupby('genre')[loop_columns].mean().round(3)

In [ ]:
game_loop_examples = features_df.loc[
    features_df['genre'].eq('game'),
    [
        'filename',
        'longest_repeated_span_ratio',
        'repeat_gap_ratio',
        'repeated_window_ratio',
        'chord_loop_closure_similarity',
        'opening_chord',
        'closing_chord',
    ],
].sort_values(
    ['longest_repeated_span_ratio', 'chord_loop_closure_similarity'],
    ascending=False,
)

game_loop_examples.head(15)